I used the `yf.download()` function from the Python package `yfinance` to obtain daily stock prices from Yahoo Finance.

I first selected four stocks for one year: Apple Inc. (`AAPL`), Tesla Inc. (`TSLA`), The Coca-Cola Company (`KO`), and Walmart Inc. (`WMT`). This was used to show the basic behavior of Universal Portfolio algorithm,the data were saved as `prices_4stocks.csv`.

Next, I expanded the dataset by using a longer time horizon for additional experiments. I hope to obtain results similar to Cover's original paper, I considered two pairs of stocks:

1. (`AAPL`) and (`KO`)
2. (`BA`) and (`PG`)

Then we also use the same dataset to study the approximation error and computational running time of the Universal Portfolio algorithm under different grid sizes.

In [ ]:
using YFinance
using DataFrames
using CSV

tickers = ["AAPL", "TSLA", "KO", "WMT"]

# Download daily adjusted closing prices from Yahoo Finance

data_list = get_prices.(tickers;startdt= "2025-01-01",enddt="2026-01-01",interval = "1d",autoadjust = true)
dfs = DataFrame.(data_list)
# Create one DataFrame with timestamp and close prices
price_df = DataFrame(timestamp=dfs[1].timestamp) #The first column represents the trading date.


for i in 1:length(tickers)
    price_df[!, tickers[i]] = dfs[i].close
end
dropmissing!(price_df)
CSV.write("prices_4stocks.csv", price_df)
println(first(price_df, 5))
println(size(price_df))

5×5 DataFrame
 Row │ timestamp            AAPL     TSLA     KO       WMT     
     │ DateTime             Float64  Float64  Float64  Float64 
─────┼─────────────────────────────────────────────────────────
   1 │ 2025-01-02T14:30:00   243.85   379.28    61.84    90.0
   2 │ 2025-01-03T14:30:00   243.36   410.44    61.75    90.78
   3 │ 2025-01-06T14:30:00   245.0    411.05    60.81    91.43
   4 │ 2025-01-07T14:30:00   242.21   394.36    60.84    90.81
   5 │ 2025-01-08T14:30:00   242.7    394.94    61.71    91.8
(250, 5)


During the data downloading process, I encountered some download errors because some stocks were not publicly listed during the selected time period. 

Therefore, I adjusted our stock universe and only kept stocks with valid historical price data over the full sample period.(We remove `TSLA` and add `BA` and `PG`) 

In [10]:
tickers = ["AAPL", "MSFT", "KO", "WMT","BA", "PG"]
aapl= DataFrame(get_prices("AAPL";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))
msft= DataFrame(get_prices("MSFT";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))
ko= DataFrame(get_prices("KO";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))
wmt= DataFrame(get_prices("WMT";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))
ba= DataFrame(get_prices("BA";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))
pg= DataFrame(get_prices("PG";startdt="2006-01-01",enddt="2026-01-01",interval="1d",autoadjust=true))

aapl= DataFrame(date=aapl.timestamp,AAPL=aapl.close)
msft= DataFrame(date=msft.timestamp,MSFT=msft.close)
ko= DataFrame(date=ko.timestamp,KO=ko.close)
wmt= DataFrame(date=wmt.timestamp,WMT=wmt.close)
ba= DataFrame(date=ba.timestamp,BA=ba.close)
pg= DataFrame(date=pg.timestamp,PG=pg.close)

# We use innerjoin to align all stocks by the same trading dates.To ensure that
# the daily return vector can be computed correctly
prices = innerjoin(aapl,msft,ko,wmt,ba,pg,on=:date)
dropmissing!(prices)

CSV.write("prices_6stocks_20years.csv", prices)

"prices_6stocks_20years.csv"